# RSNA 2018 — Pipeline de Modelagem, Avaliação e Detecção

Este notebook executa o fluxo experimental pesado do baseline clássico: **validação das features, classificação, ablação, avaliação e detecção/localização**.
Os dados derivados ficam no Google Drive; o GitHub contém o código oficial e reproduzível.

In [2]:
BRANCH = "main"
!rm -rf ap1-rsna-pneumonia-baseline
!git clone -q -b {BRANCH} https://github.com/erik-hp/ap1-rsna-pneumonia-baseline.git
%cd ap1-rsna-pneumonia-baseline
!pip -q install -r requirements.txt
!pip -q install -e .
!pytest -q

/content/ap1-rsna-pneumonia-baseline
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 684.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.3/342.3 kB 31.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency confli

## 1. Montar o Drive

A pasta esperada é `MyDrive/RSNA_TP1/data/processed/`.

In [5]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/RSNA_TP1")
PROCESSED = BASE / "data" / "processed"
OUT = BASE / "modelagem_avaliacao"
OUT.mkdir(parents=True, exist_ok=True)
FEATURES_ALL = PROCESSED / "features_all.csv"
FEATURES_REGIONS = PROCESSED / "features_regions.csv"
GROUND_TRUTH = PROCESSED / "ground_truth_boxes.csv"
IMAGE_MANIFEST = PROCESSED / "image_manifest.csv"
for p in [FEATURES_ALL, FEATURES_REGIONS, GROUND_TRUTH, IMAGE_MANIFEST]:
    print(("OK      " if p.exists() else "FALTANDO "), p)

Mounted at /content/drive
OK       /content/drive/MyDrive/RSNA_TP1/data/processed/features_all.csv
OK       /content/drive/MyDrive/RSNA_TP1/data/processed/features_regions.csv
OK       /content/drive/MyDrive/RSNA_TP1/data/processed/ground_truth_boxes.csv
OK       /content/drive/MyDrive/RSNA_TP1/data/processed/image_manifest.csv


## 2. Validar a matriz de classificação

In [ ]:
!python scripts/validate_features.py --features "{FEATURES_ALL}" --auto-ablation

{
  "rows": 500,
  "unique_groups": 500,
  "classes": {
    "0": 387,
    "1": 113
  },
  "groups_per_class": {
    "0": 387,
    "1": 113
  },
  "positive_prevalence": 0.226,
  "feature_count_total": 346,
  "feature_counts_by_family": {
    "HOG": 324,
    "LBP": 10,
    "GLCM": 12,
    "HOG+LBP": 334,
    "HOG+GLCM": 336,
    "LBP+GLCM": 22,
    "HOG+LBP+GLCM": 346
  },
  "duplicate_full_rows": 0,
  "rows_beyond_one_per_group": 0,
  "has_candidate_boxes": false
}


## 3. Experimento principal — classificação por radiografia

Roda Dummy, SVM linear/RBF, Random Forest e HistGradientBoosting com nested CV 5×3 e todas as combinações HOG/LBP/GLCM.

In [8]:
CLASS_RESULTS = OUT / "classification" / "results"
CLASS_FIGURES = OUT / "classification" / "figures"
!python scripts/run_experiment.py --features "{FEATURES_ALL}" --auto-ablation --outer-splits 5 --inner-splits 3 --seed 42 --output-dir "{CLASS_RESULTS}" --figures-dir "{CLASS_FIGURES}" --plots best


Experimento concluído sem usar o teste oficial do Kaggle.
Resultados: /content/drive/MyDrive/RSNA_TP1/modelagem_avaliacao/classification/results
Figuras: /content/drive/MyDrive/RSNA_TP1/modelagem_avaliacao/classification/figures

Auditoria de entrada:
{
  "rows": 500,
  "unique_groups": 500,
  "classes": {
    "0": 387,
    "1": 113
  },
  "groups_per_class": {
    "0": 387,
    "1": 113
  },
  "positive_prevalence": 0.226,
  "feature_count_total": 346,
  "feature_counts_by_family": {
    "HOG": 324,
    "LBP": 10,
    "GLCM": 12,
    "HOG+LBP": 334,
    "HOG+GLCM": 336,
    "LBP+GLCM": 22,
    "HOG+LBP+GLCM": 346
  },
  "duplicate_full_rows": 0,
  "rows_beyond_one_per_group": 0,
  "has_candidate_boxes": false
}

Resumo (média ± desvio-padrão):
feature_family                  model       auc_roc        auc_pr   sensitivity   specificity            f1 balanced_accuracy
          GLCM                  dummy 0.500 ± 0.000 0.226 ± 0.049 0.000 ± 0.000 1.000 ± 0.000 0.000 ± 0.000     0.500 

## 4. Sensibilidade ao tratamento de desbalanceamento (opcional, mas já preparado)

Execute se houver tempo para comparar o estudo principal com os mesmos modelos sem `class_weight`.

In [ ]:
NO_WEIGHT_RESULTS = OUT / "classification_no_class_weight" / "results"
NO_WEIGHT_FIGURES = OUT / "classification_no_class_weight" / "figures"
# Descomente para executar:
# !python scripts/run_experiment.py --features "{FEATURES_ALL}" --auto-ablation --class-weight none --outer-splits 5 --inner-splits 3 --seed 42 --output-dir "{NO_WEIGHT_RESULTS}" --figures-dir "{NO_WEIGHT_FIGURES}" --plots none

## 5. Detecção/localização — etapa pesada

Usa regiões candidatas, separação OOF por paciente, undersampling apenas no treino, HistGradientBoosting e NMS.

**Use o `features_regions.csv` regenerado com as 175 regiões da versão atual.**

In [6]:
DETECTION_RESULTS = OUT / "detection"
!python scripts/run_detection_experiment.py --features-regions "{FEATURES_REGIONS}" --ground-truth "{GROUND_TRUTH}" --image-manifest "{IMAGE_MANIFEST}" --model hist_gradient_boosting --family "HOG+LBP+GLCM" --outer-splits 5 --inner-splits 3 --max-neg-pos-ratio 3 --nms-iou 0.30 --top-k 20 --seed 42 --output-dir "{DETECTION_RESULTS}"

Fold 1/5: train=4,648 regiões, test=17,500, AUC-PR regional=0.1129
Fold 2/5: train=4,528 regiões, test=17,500, AUC-PR regional=0.1509
Fold 3/5: train=4,776 regiões, test=17,500, AUC-PR regional=0.1405
Fold 4/5: train=4,868 regiões, test=17,500, AUC-PR regional=0.1548
Fold 5/5: train=4,684 regiões, test=17,500, AUC-PR regional=0.1326
OOF regional preservado. Iniciando NMS, mAP e FROC com cálculo FROC incremental.

Detecção OOF concluída.
{
  "ap_iou_0.50": 0.02297955080153948,
  "ap_iou_0.55": 0.010199428035446895,
  "ap_iou_0.60": 0.004394276514844817,
  "ap_iou_0.65": 0.000487634925760576,
  "ap_iou_0.70": 9.209672401695809e-05,
  "ap_iou_0.75": 0.0,
  "ap_iou_0.80": 0.0,
  "ap_iou_0.85": 0.0,
  "ap_iou_0.90": 0.0,
  "ap_iou_0.95": 0.0,
  "map_50_95": 0.003815298700160873,
  "froc_sens_at_0.125_fp_per_image": 0.05357142857142857,
  "froc_sens_at_0.25_fp_per_image": 0.08928571428571429,
  "froc_sens_at_0.5_fp_per_image": 0.1130952380952381,
  "froc_sens_at_1_fp_per_image": 0.125,
  "fr

## 6. Conferir os resultados para o artigo

In [9]:
import json, pandas as pd
print("\n=== CLASSIFICAÇÃO ===")
display(pd.read_csv(CLASS_RESULTS / "summary_formatted.csv"))
print("\n=== DETECÇÃO ===")
print(json.dumps(json.loads((DETECTION_RESULTS / "detection_metrics.json").read_text()), indent=2))
print("\nArquivos finais em:", OUT)


=== CLASSIFICAÇÃO ===


,feature_family,model,auc_roc,auc_pr,sensitivity,specificity,f1,balanced_accuracy
0,GLCM,dummy,0.500 ± 0.000,0.226 ± 0.049,0.000 ± 0.000,1.000 ± 0.000,0.000 ± 0.000,0.500 ± 0.000
1,GLCM,hist_gradient_boosting,0.566 ± 0.102,0.307 ± 0.112,0.291 ± 0.155,0.785 ± 0.039,0.286 ± 0.140,0.538 ± 0.089
2,GLCM,random_forest,0.597 ± 0.097,0.329 ± 0.113,0.117 ± 0.105,0.907 ± 0.051,0.151 ± 0.113,0.512 ± 0.038
3,GLCM,svm_linear,0.669 ± 0.083,0.404 ± 0.140,0.507 ± 0.092,0.699 ± 0.091,0.399 ± 0.109,0.603 ± 0.062
4,GLCM,svm_rbf,0.662 ± 0.076,0.388 ± 0.134,0.565 ± 0.078,0.692 ± 0.048,0.430 ± 0.109,0.628 ± 0.057
5,HOG,dummy,0.500 ± 0.000,0.226 ± 0.049,0.000 ± 0.000,1.000 ± 0.000,0.000 ± 0.000,0.500 ± 0.000
6,HOG,hist_gradient_boosting,0.689 ± 0.042,0.402 ± 0.088,0.232 ± 0.052,0.900 ± 0.028,0.290 ± 0.062,0.566 ± 0.028
7,HOG,random_forest,0.741 ± 0.061,0.457 ± 0.141,0.070 ± 0.065,0.978 ± 0.025,0.109 ± 0.101,0.524 ± 0.023
8,HOG,svm_linear,0.627 ± 0.060,0.388 ± 0.090,0.414 ± 0.087,0.748 ± 0.043,0.359 ± 0.078,0.581 ± 0.041
9,HOG,svm_rbf,0.734 ± 0.067,0.472 ± 0.144,0.328 ± 0.195,0.870 ± 0.086,0.319 ± 0.191,0.599 ± 0.065



=== DETECÇÃO ===
{
  "ap_iou_0.50": 0.02297955080153948,
  "ap_iou_0.55": 0.010199428035446895,
  "ap_iou_0.60": 0.004394276514844817,
  "ap_iou_0.65": 0.000487634925760576,
  "ap_iou_0.70": 9.209672401695809e-05,
  "ap_iou_0.75": 0.0,
  "ap_iou_0.80": 0.0,
  "ap_iou_0.85": 0.0,
  "ap_iou_0.90": 0.0,
  "ap_iou_0.95": 0.0,
  "map_50_95": 0.003815298700160873,
  "froc_sens_at_0.125_fp_per_image": 0.05357142857142857,
  "froc_sens_at_0.25_fp_per_image": 0.08928571428571429,
  "froc_sens_at_0.5_fp_per_image": 0.1130952380952381,
  "froc_sens_at_1_fp_per_image": 0.125,
  "froc_sens_at_2_fp_per_image": 0.15476190476190477,
  "froc_sens_at_4_fp_per_image": 0.20238095238095238,
  "froc_sens_at_8_fp_per_image": 0.23809523809523808
}

Arquivos finais em: /content/drive/MyDrive/RSNA_TP1/modelagem_avaliacao


## Saídas preservadas no Drive\n\n- `modelagem_avaliacao/classification/results/`: métricas, ablação, parâmetros e erros;\n- `modelagem_avaliacao/classification/figures/`: ROC, PR e matriz de confusão;\n- `modelagem_avaliacao/detection/`: OOF regional, predições após NMS, mAP e FROC;\n- `modelagem_avaliacao/report_figures/`: figuras consolidadas para o relatório (Top 10 PR-AUC, trade-off sensibilidade × especificidade e FROC).\n\nNão use resultados de smoke test no artigo.

## 7. Gerar figuras finais para o relatório\n\nEsta etapa é **aditiva** e usa somente os arquivos de resultados já produzidos nas etapas anteriores. Ela não treina novamente os modelos nem altera as métricas. As figuras são salvas no Google Drive em `modelagem_avaliacao/report_figures/` e exibidas abaixo.\n

In [ ]:
REPORT_FIGURES = OUT / "report_figures"\n!python scripts/generate_report_figures.py --classification-results "{CLASS_RESULTS}" --detection-results "{DETECTION_RESULTS}" --output-dir "{REPORT_FIGURES}"\n\nfrom IPython.display import Image, display\nfor figure_name in [\n    "classificacao_top10_prauc.png",\n    "tradeoff_sensibilidade_especificidade.png",\n    "froc_deteccao.png",\n]:\n    figure_path = REPORT_FIGURES / figure_name\n    print(figure_path)\n    display(Image(filename=str(figure_path)))\n